### Creates `openalex.authors.parsed_names_normalized` table

Reads from `parsed_names_lookup` (already parsed first/middle/last) and adds:
- **Romanization**: `unidecode` for CJK/Cyrillic/Arabic → ASCII
- **ALL CAPS surname detection**: fixes French-style "DUPONT Jean" parsing
- **Middle initials detection**: periods, caps, or short strings → initials
- **Always-present initial columns**: `first_initial`, `middle_initials`

Job: #105.8 normalize-author-names

In [ ]:
%pip install nameparser unidecode

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS identifier('openalex' || :env_suffix || '.authors.parsed_names_normalized') (
  raw_author_name STRING,
  parsed_name STRUCT<
      title: STRING,
      first: STRING,
      middle: STRING,
      last: STRING,
      suffix: STRING,
      nickname: STRING
  >,
  created_datetime TIMESTAMP,
  normalized_first STRING,
  first_initial STRING,
  normalized_middle STRING,
  middle_initials STRING,
  middle_initial_count INT,
  normalized_last STRING
)
USING DELTA
CLUSTER BY (raw_author_name)

#### Imports, constants, two-letter names list

In [ ]:
import unicodedata
import re

from unidecode import unidecode
from nameparser import HumanName

import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

dbutils.widgets.text("env_suffix", "", "Environment Suffix")
env_suffix = dbutils.widgets.get("env_suffix")

# Top two-letter first names from name_freq_first_unified (freq > 0, ASCII, has vowel).
# Used to prevent misclassifying real names like "Li" or "Bo" as initials.
# Source: SELECT name, freq FROM openalex.authors.name_freq_first_unified
#         WHERE LENGTH(name) = 2 ORDER BY freq DESC
# Filtered: ASCII only, contains at least one vowel (aeiou).
TWO_LETTER_NAMES = {
    'li', 'bo', 'yi', 'yu', 'ko', 'so', 'qi', 'yo', 'ma', 'xu', 'na', 'lu',
    'ke', 'ha', 'mi', 'ye', 'su', 'ei', 'ta', 'xi', 'he', 'ao', 'ia', 'az',
    'di', 'ji', 'hu', 'fu', 'jo', 'ka', 'do', 'en', 'wu', 'ed', 'ya', 'ki',
    'ak', 'oh', 'vo', 'ag', 'ur', 'ej', 'mu', 'le', 'ge', 'al', 'se', 'ah',
    'zo', 'vi', 'um', 'oo', 'ri', 'ga', 'ja', 'je', 'at', 'ju', 'da', 'an',
    'ca', 'eh', 'vu', 'ug', 'ea', 'or', 'pa', 'ay', 'is', 'ne', 'av', 'fe',
    'co', 'et', 'ru', 'aj', 'ze', 'de', 'mo', 'ni', 'si', 'pu', 'fa', 'za',
    'el', 'xo', 'ku', 'ut', 'wa', 'eg', 'uy', 'iu', 'ie', 'pe', 'oz', 'ae',
    'ho', 'cu', 'ax', 'nu', 'ez', 'ai', 'em', 'om', 'ce', 'tu', 'zi', 'ro',
    'bi', 'ad', 'po', 'ou', 'ui', 'ua', 'io', 'on', 'go', 'og', 'ar', 'ba',
    'gu', 'te', 'ra', 'la', 'ac', 'er', 'bu', 'ti', 'gi', 'du', 'ci', 'es',
    'ev', 'pi', 'va', 'zu', 'id', 've', 'oi', 'ol', 'be', 'ot', 'ec', 'ef',
    'ud', 'iv', 'ii', 'wo', 'wi', 'ee', 'iz', 'od', 'xa', 'no', 'oa', 'me',
    'uz', 'ul', 'oy', 'uu', 'xe', 'as', 'il', 'us', 'un', 'ok', 're',
}

#### Core functions

In [ ]:
def normalize_name_part(s):
    """Romanize and clean a name part to ASCII lowercase.
    
    Pipeline: unidecode → NFD strip diacritics → lowercase → drop periods
    → keep only letters/spaces/hyphens → collapse whitespace.
    """
    if not s or not isinstance(s, str):
        return ''
    s = s.strip()
    if not s:
        return ''
    # Romanize non-Latin scripts (CJK, Cyrillic, Arabic, etc.)
    result = unidecode(s)
    # Strip remaining diacritics via NFD decomposition
    result = unicodedata.normalize('NFD', result)
    result = ''.join(c for c in result if unicodedata.category(c) != 'Mn')
    # Lowercase and remove periods
    result = result.lower().replace('.', '')
    # Keep only letters, spaces, and hyphens
    result = re.sub(r'[^a-z\s-]', '', result)
    # Collapse whitespace
    result = re.sub(r'\s+', ' ', result).strip()
    return result


def detect_all_caps_surname(raw_name):
    """Detect French-style ALL CAPS surname in a raw author name.
    
    Rules (ALL must be true):
    - Exactly one token is ALL CAPS
    - That token is purely alphabetic (no periods, digits, hyphens)
    - That token has length >= 3
    - That token contains at least one vowel (AEIOU)
    
    Returns (first_raw, middle_raw, last_raw) if detected, else None.
    The ALL CAPS token becomes last; remaining tokens become first + middle.
    """
    if not raw_name or not isinstance(raw_name, str):
        return None
    tokens = raw_name.split()
    if len(tokens) < 2:
        return None

    caps_indices = [
        i for i, t in enumerate(tokens)
        if t.isalpha() and t.isupper() and len(t) >= 3 and re.search(r'[AEIOU]', t)
    ]
    if len(caps_indices) != 1:
        return None

    caps_idx = caps_indices[0]
    surname = tokens[caps_idx]
    remaining = [t for i, t in enumerate(tokens) if i != caps_idx]

    if len(remaining) == 1:
        return (remaining[0], '', surname)
    else:
        return (remaining[0], ' '.join(remaining[1:]), surname)


def is_middle_initials(norm_middle, raw_author_name):
    """Determine if a parsed middle name represents initials.
    
    Three rules (any triggers initials):
    1. Period-separated in raw name (e.g. 'H.W.', 'J.P.R.')
    2. Middle tokens are ALL CAPS in raw name, but whole name isn't all caps
    3. Length < 3 (after stripping spaces), unless in TWO_LETTER_NAMES
    
    Args:
        norm_middle: normalized middle name (lowercase, ASCII)
        raw_author_name: original raw string with casing intact.
            When CAPS correction fires, caller passes the reconstructed name
            (surname removed) so rule 2 checks the right tokens.
    """
    if not norm_middle:
        return False

    m_stripped = re.sub(r'\s', '', norm_middle)
    if not m_stripped:
        return False

    raw = raw_author_name if isinstance(raw_author_name, str) else ''

    # Rule 1: Check raw name for period-separated initials
    if raw and re.search(r'(?<!\w)[A-Za-z]\.\s*[A-Za-z]\.', raw):
        return True

    # Rule 2: Middle tokens in raw name are ALL CAPS, but whole name isn't
    if raw and not raw.isupper():
        tokens = raw.split()
        if len(tokens) >= 3:
            middle_tokens = tokens[1:-1]
            if middle_tokens and all(t.isupper() for t in middle_tokens):
                return True

    # Rule 3: Short middle (< 3 chars), unless it's a known two-letter name
    if len(m_stripped) < 3:
        if len(m_stripped) == 2 and m_stripped in TWO_LETTER_NAMES:
            return False  # real name, not initials
        return True  # single char or unknown 2-char → initials

    return False


def process_row(raw_name, parsed_first, parsed_middle, parsed_last):
    """Process one row: normalize names, detect initials, correct ALL CAPS surnames.
    
    Returns dict with: normalized_first, first_initial, normalized_middle,
    middle_initials, middle_initial_count, normalized_last.
    """
    # Sanitize inputs — pandas NaN is truthy but not a string
    raw_name = raw_name if isinstance(raw_name, str) else ''
    parsed_first = parsed_first if isinstance(parsed_first, str) else ''
    parsed_middle = parsed_middle if isinstance(parsed_middle, str) else ''
    parsed_last = parsed_last if isinstance(parsed_last, str) else ''

    # Step 1: Check for ALL CAPS surname in raw name
    caps_result = detect_all_caps_surname(raw_name)
    if caps_result:
        first_src, middle_src, last_src = caps_result
        # Reconstruct raw name without the CAPS surname so is_middle_initials
        # checks casing on the remaining tokens, not the removed surname.
        parts = [first_src] + ([middle_src] if middle_src else [])
        raw_for_initials = ' '.join(parts)
    else:
        first_src = parsed_first
        middle_src = parsed_middle
        last_src = parsed_last
        raw_for_initials = raw_name

    # Step 2: Normalize all parts (unidecode + clean)
    norm_first = normalize_name_part(first_src)
    norm_middle = normalize_name_part(middle_src)
    norm_last = normalize_name_part(last_src)

    # Step 3: First initial — always the first char
    first_initial = norm_first[0] if norm_first else ''

    # Step 4: Middle initials detection (uses norm_middle + raw_for_initials,
    # so it works correctly for both CAPS-corrected and normal rows)
    if not norm_middle:
        mid_initials = None
        mid_count = 0
    elif is_middle_initials(norm_middle, raw_for_initials):
        # It's initials — the whole normalized middle IS the initials
        mid_initials = re.sub(r'[^a-z]', '', norm_middle)
        mid_count = len(mid_initials) if mid_initials else 0
    else:
        # Full name — derive single initial
        mid_initials = norm_middle[0]
        mid_count = 1

    return {
        'normalized_first': norm_first or None,
        'first_initial': first_initial or None,
        'normalized_middle': norm_middle or None,
        'middle_initials': mid_initials,
        'middle_initial_count': mid_count,
        'normalized_last': norm_last or None,
    }

#### Pandas UDF for Spark

In [ ]:
norm_schema = StructType([
    StructField('normalized_first', StringType(), True),
    StructField('first_initial', StringType(), True),
    StructField('normalized_middle', StringType(), True),
    StructField('middle_initials', StringType(), True),
    StructField('middle_initial_count', IntegerType(), True),
    StructField('normalized_last', StringType(), True),
])

# Column names in schema order — Spark matches Pandas UDF output by POSITION, not name
_NORM_COLUMNS = [f.name for f in norm_schema.fields]


@F.pandas_udf(norm_schema)
def normalize_names_batch(
    raw_names: pd.Series,
    firsts: pd.Series,
    middles: pd.Series,
    lasts: pd.Series,
) -> pd.DataFrame:
    """Vectorized UDF: normalize and classify name parts."""
    results = [
        process_row(rn, f, m, l)
        for rn, f, m, l in zip(raw_names, firsts, middles, lasts)
    ]
    return pd.DataFrame(results, columns=_NORM_COLUMNS)

#### Build the table

In [ ]:
source_table = f"openalex{env_suffix}.authors.parsed_names_lookup"
target_table = f"openalex{env_suffix}.authors.parsed_names_normalized"

source_df = spark.table(source_table)
total_count = source_df.count()
print(f"Source table has {total_count:,} rows")

# Repartition for parallelism (small cluster: 2-4 workers × 4 cores = 8-16 cores)
source_df = source_df.repartition(100)

# Apply normalization UDF
result_df = source_df.withColumn(
    "norm",
    normalize_names_batch(
        F.col("raw_author_name"),
        F.col("parsed_name.first"),
        F.col("parsed_name.middle"),
        F.col("parsed_name.last"),
    ),
).select(
    "raw_author_name",
    "parsed_name",
    "created_datetime",
    F.col("norm.normalized_first"),
    F.col("norm.first_initial"),
    F.col("norm.normalized_middle"),
    F.col("norm.middle_initials"),
    F.col("norm.middle_initial_count"),
    F.col("norm.normalized_last"),
)

# Write as new table (overwrite for initial build; switch to append for incremental)
result_df.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(target_table)

written_count = spark.table(target_table).count()
print(f"Wrote {written_count:,} rows to {target_table}")
print(f"Row count match: {written_count == total_count}")

#### Verification

In [ ]:
# Test 1: Non-ASCII eliminated
non_ascii = spark.sql(f"""
    SELECT COUNT(*) AS non_ascii_count FROM {target_table}
    WHERE normalized_last RLIKE '[^\\x00-\\x7F]'
       OR normalized_first RLIKE '[^\\x00-\\x7F]'
       OR normalized_middle RLIKE '[^\\x00-\\x7F]'
""").collect()[0]['non_ascii_count']
print(f"Test 1 - Non-ASCII in normalized columns: {non_ascii} (should be 0)")

In [ ]:
# Test 2: first_initial and middle_initials always populated when parent exists
missing_first_initial = spark.sql(f"""
    SELECT COUNT(*) AS c FROM {target_table}
    WHERE normalized_first IS NOT NULL AND first_initial IS NULL
""").collect()[0]['c']
missing_mid_initial = spark.sql(f"""
    SELECT COUNT(*) AS c FROM {target_table}
    WHERE normalized_middle IS NOT NULL AND middle_initials IS NULL
""").collect()[0]['c']
print(f"Test 2 - Missing first_initial: {missing_first_initial} (should be 0)")
print(f"Test 2 - Missing middle_initials: {missing_mid_initial} (should be 0)")

In [ ]:
# Test 3: Spot-check known cases
spot_checks = spark.sql(f"""
    SELECT raw_author_name, normalized_first, first_initial,
           normalized_middle, middle_initials, middle_initial_count, normalized_last
    FROM {target_table}
    WHERE raw_author_name IN (
        'George H.W. Bush', 'J.R.R. Tolkien', 'C.S. Lewis', 'John William Smith'
    )
""")
spot_checks.show(truncate=False)

In [ ]:
# Test 5: ALL CAPS surname detection
caps_checks = spark.sql(f"""
    SELECT raw_author_name, normalized_first, normalized_last
    FROM {target_table}
    WHERE raw_author_name IN ('DUPONT Jean', 'Marie CURIE', 'AB Smith', 'JONES SMITH')
""")
caps_checks.show(truncate=False)

In [ ]:
# Summary stats
spark.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN LENGTH(normalized_first) = 1 THEN 1 ELSE 0 END) AS first_is_initial_count,
        SUM(CASE WHEN middle_initial_count > 1 THEN 1 ELSE 0 END) AS multi_initial_middle_count,
        SUM(CASE WHEN normalized_middle IS NOT NULL
                  AND normalized_middle != middle_initials THEN 1 ELSE 0 END) AS full_middle_name_count,
        SUM(CASE WHEN normalized_middle IS NULL THEN 1 ELSE 0 END) AS no_middle_count
    FROM {target_table}
""").show(truncate=False)